Load the data

In [1]:
#  Loading the data with all 200 features (for 200x200 data) - alternative loading method

from scipy.io import loadmat
import torch
import numpy as np

filepath = "Input Data\FC1.mat"
fc_mat = loadmat(filepath)
fc_mat = fc_mat['FC1']  # Extract the FC1 variable from the loaded .mat file

print(fc_mat.shape)  # Should print (200, 200, N_subjects)

fc_mat_inv = np.transpose(fc_mat, (2, 0, 1))  # Transpose to (N_subjects, 200, 200)
fc_mat_inv = fc_mat_inv[:, None, :, :]  # Add a channel dimension to get (N_subjects, 1, 200, 200)
print(fc_mat_inv.shape)  # Should print (N_subjects, 1, 200, 200)

X = torch.from_numpy(fc_mat_inv).float()  # Convert to PyTorch tensor
print("Data shape:", X.shape)  # Should print (N_subjects, 1, 200, 200)

# Expected shape is (Batch size, channels, height, width)

(200, 200, 72)
(72, 1, 200, 200)
Data shape: torch.Size([72, 1, 200, 200])


In [ ]:
def check_zero_mean_unit_range(data, atol=1e-5):
    if hasattr(data, "detach"):
        arr = data.detach().cpu().numpy()
    else:
        arr = np.asarray(data)

    mean_val = arr.mean()
    min_val = arr.min()
    max_val = arr.max()

    is_zero_mean = np.isclose(mean_val, 0.0, atol=atol)
    has_minus_one = np.isclose(min_val, -1.0, atol=atol)
    has_plus_one = np.isclose(max_val, 1.0, atol=atol)

    print(f"Doing checks for zero mean and unit range:")
    print(f"mean: {mean_val:.6f}")
    print(f"standard deviation: {arr.std():.6f}")
    print(f"min:  {min_val:.6f}")
    print(f"max:  {max_val:.6f}")
    print(f"zero mean: {is_zero_mean}")
    print(f"min == -1: {has_minus_one}")
    print(f"max ==  1: {has_plus_one}")

    return is_zero_mean and has_minus_one and has_plus_one

# Example:
# check_zero_mean_unit_range(X)

def z_normalize(data, eps=1e-8):
    if hasattr(data, "detach"):
        mean = data.mean(dim=(1, 2, 3), keepdim=True)
        std = data.std(dim=(1, 2, 3), keepdim=True, unbiased=False).clamp_min(eps)
        return (data - mean) / std

    arr = np.asarray(data)
    mean = arr.mean(axis=tuple(range(1, arr.ndim)), keepdims=True)
    std = arr.std(axis=tuple(range(1, arr.ndim)), keepdims=True)
    std = np.maximum(std, eps)
    return (arr - mean) / std


In [8]:
check_zero_mean_unit_range(X)
z_normalized_X = z_normalize(X)
check_zero_mean_unit_range(z_normalized_X)

Doing checks for zero mean and unit range:
mean: 0.008007
min:  -0.825187
max:  1.000000
zero mean: False
min == -1: False
max ==  1: True
Doing checks for zero mean and unit range:
mean: -0.000000
min:  -2.785055
max:  3.682692
zero mean: True
min == -1: False
max ==  1: False


False

In [1]:
import pandas as pd

def read_csv_and_column_min_max(csv_path):
    df = pd.read_csv(csv_path)
    min_max = pd.DataFrame({
        "min": df.min(numeric_only=True),
        "max": df.max(numeric_only=True)
    })
    return df, min_max

# Example:
# df, column_stats = read_csv_and_column_min_max("your_file.csv")
# print(column_stats)

In [3]:
csv_path = "C:\Mats og Odd Arne\Prosjektoppgave\ISC_data\Beh.csv"

df, column_stats = read_csv_and_column_min_max(csv_path)
for value in  column_stats.itertuples():
    print(f"{value.Index}: min={value.min}, max={value.max}")

Subject: min=11001.0, max=12334.0
Group: min=1.0, max=2.0
Age: min=19.0, max=82.0
Order: min=1.0, max=141.0
Sex: min=1.0, max=2.0
Relationshipstatus: min=1.0, max=5.0
Avg_Sleep: min=4.0, max=10.5
EnglishYearsSpeaking: min=0.0, max=43.0
YearsEducation: min=0.0, max=29.0
PsychDiagnosis: min=0.0, max=1.0
Medicine: min=0.0, max=1.0
NeuroImpairment: min=0.0, max=1.0
EV_base: min=1.0, max=9.0
EV_neu: min=1.0, max=9.0
EV_neg: min=1.0, max=9.0
ER_base: min=1.0, max=9.0
ER_neu: min=1.0, max=9.0
ER_neg: min=1.0, max=9.0
dEV_neu: min=-4.0, max=6.0
dEV_neg: min=-8.0, max=6.0
dER_neu: min=-3.0, max=5.0
dER_neg: min=-4.0, max=8.0
Q1_DASSDepression: min=0.0, max=18.0
Q1_DASSAnxiety: min=0.0, max=14.0
Q1_DASSStress: min=0.0, max=19.0
Q2_DERSScore: min=47.0, max=119.0
Q2_DERSNONACCEPT: min=6.0, max=23.0
Q2_DERSGOALS: min=5.0, max=25.0
Q2_DERSIMPULSE: min=6.0, max=22.0
Q2_DERSAWARENESS: min=6.0, max=28.0
Q2_DERSSTRATEGIES: min=8.0, max=31.0
Q2_DERSCLARITY: min=5.0, max=21.0
Q3_GHQ28Score: min=31.0, max=

<>:1: SyntaxWarning: invalid escape sequence '\M'
<>:1: SyntaxWarning: invalid escape sequence '\M'
C:\Users\oddafo\AppData\Local\Temp\ipykernel_4236\3478438818.py:1: SyntaxWarning: invalid escape sequence '\M'
  csv_path = "C:\Mats og Odd Arne\Prosjektoppgave\ISC_data\Beh.csv"


In [ ]:
import scipy.io as sio
import pandas as pd
import numpy as np
import re


def extract_id_from_mat_name(text):
    """
    Extract subject ID from text between 'sub-' and '-task'.
    Example:
        'something_sub-0123-task-rest_run-1_bold.mat' -> '0123'
    """
    text = str(text).strip()
    match = re.search(r"sub-(.*?)-task", text)
    return match.group(1).strip() if match else None


def normalize_id(x, remove_leading_zeros=False):
    """
    Convert IDs to a common format so strings/numbers compare correctly.
    """
    x = str(x).strip()

    # Convert things like 123.0 -> 123
    if re.fullmatch(r"\d+\.0", x):
        x = x[:-2]

    if remove_leading_zeros and x.isdigit():
        x = str(int(x))

    return x


def flatten_subject_names(arr):
    """
    Flatten MATLAB subject_names into a list of strings.
    Works for common scipy.io.loadmat outputs.
    """
    arr = np.asarray(arr)
    values = []

    for item in arr.flat:
        if isinstance(item, np.ndarray):
            if item.size == 1:
                values.append(str(item.item()).strip())
            else:
                values.append("".join(str(x) for x in item.flat).strip())
        else:
            values.append(str(item).strip())

    return values


def load_ids_from_mat(mat_file, remove_leading_zeros=False):
    """
    Load subject IDs from subject_names in a .mat file.
    """
    mat_data = sio.loadmat(mat_file)

    if "subject_names" not in mat_data:
        raise ValueError(f"'subject_names' not found in {mat_file}")

    subject_names = flatten_subject_names(mat_data["subject_names"])

    ids = set()
    for name in subject_names:
        sid = extract_id_from_mat_name(name)
        if sid is not None:
            ids.add(normalize_id(sid, remove_leading_zeros))

    return ids


def load_ids_from_csv(csv_file, remove_leading_zeros=False):
    """
    Load IDs from a CSV file.
    This searches all cells and keeps values that look like IDs.
    If your CSV has a specific ID column, I can make this stricter.
    """
    df = pd.read_csv(csv_file)

    ids = set()

    for col in df.columns:
        for value in df[col].dropna():
            val = normalize_id(value, remove_leading_zeros)
            ids.add(val)

    return ids


def compare_two_mats_against_csv(mat_file_1, mat_file_2, csv_file, remove_leading_zeros=False):
    """
    Compare the COMBINED subject IDs from two MAT files against one CSV file.
    Returns:
        only_in_combined_mats
        only_in_csv
    """
    mat_ids_1 = load_ids_from_mat(mat_file_1, remove_leading_zeros)
    mat_ids_2 = load_ids_from_mat(mat_file_2, remove_leading_zeros)

    combined_mat_ids = mat_ids_1.union(mat_ids_2)
    csv_ids = load_ids_from_csv(csv_file, remove_leading_zeros)

    only_in_combined_mats = sorted(combined_mat_ids - csv_ids)
    only_in_csv = sorted(csv_ids - combined_mat_ids)

    print(f"\nTotal IDs in {mat_file_1}: {len(mat_ids_1)}")
    print(f"Total IDs in {mat_file_2}: {len(mat_ids_2)}")
    print(f"Total unique IDs in combined MAT files: {len(combined_mat_ids)}")
    print(f"Total IDs in CSV: {len(csv_ids)}")

    print("\nIDs in combined MAT files but not in CSV:")
    for sid in only_in_combined_mats:
        print(sid)

    print("\nIDs in CSV but not in combined MAT files:")
    for sid in only_in_csv:
        print(sid)

    return only_in_combined_mats, only_in_csv

In [38]:
Included_subjects_path = "Dummy Test Data\Included subjects.csv"
matlab_path_young = "Input Data\YoungAge_combined_run1_400_noZ.mat"
matlab_path_old = "Input Data\OldAge_combined_run1_400_noZ.mat"


missing = compare_two_mats_against_csv(
    mat_file_1=matlab_path_young,
    mat_file_2=matlab_path_old,
    csv_file=Included_subjects_path
)

<>:1: SyntaxWarning: invalid escape sequence '\I'
<>:2: SyntaxWarning: invalid escape sequence '\Y'
<>:3: SyntaxWarning: invalid escape sequence '\O'
<>:1: SyntaxWarning: invalid escape sequence '\I'
<>:2: SyntaxWarning: invalid escape sequence '\Y'
<>:3: SyntaxWarning: invalid escape sequence '\O'
C:\Users\oddafo\AppData\Local\Temp\ipykernel_4236\1661315378.py:1: SyntaxWarning: invalid escape sequence '\I'
  Included_subjects_path = "Dummy Test Data\Included subjects.csv"
C:\Users\oddafo\AppData\Local\Temp\ipykernel_4236\1661315378.py:2: SyntaxWarning: invalid escape sequence '\Y'
  matlab_path_young = "Input Data\YoungAge_combined_run1_400_noZ.mat"
C:\Users\oddafo\AppData\Local\Temp\ipykernel_4236\1661315378.py:3: SyntaxWarning: invalid escape sequence '\O'
  matlab_path_old = "Input Data\OldAge_combined_run1_400_noZ.mat"
C:\Users\oddafo\AppData\Local\Temp\ipykernel_4236\1661315378.py:1: SyntaxWarning: invalid escape sequence '\I'
  Included_subjects_path = "Dummy Test Data\Included 

NameError: name 'compare_two_mats_against_csv' is not defined

In [ ]:
Included_subjects_path = "Dummy Test Data\Included subjects.csv"

# Read matlab file
mat_data = sio.loadmat(matlab_path)
print(mat_data.keys())  # Check available keys in the .mat file

subject_names = mat_data.get("subject_names", [])
print(subject_names.shape)

subject_names_id = [extract_id(name) for name in flatten_mat_strings(subject_names)]
print(len(subject_names_id))



dict_keys(['__header__', '__version__', '__globals__', 'run1_data', 'subject_names'])
(72,)
72


<>:1: SyntaxWarning: invalid escape sequence '\I'
<>:1: SyntaxWarning: invalid escape sequence '\I'
C:\Users\oddafo\AppData\Local\Temp\ipykernel_4236\3819817200.py:1: SyntaxWarning: invalid escape sequence '\I'
  Included_subjects_path = "Dummy Test Data\Included subjects.csv"
